# Accessing Data

The data folder is a python module. Comprehensive documentation in `README.md`. This is a series of quick examples to get you up and running.

In [ ]:
# when importing modules, python by default looks for installed system modules
# and files in the cwd.
# these extra two lines add the project root to the list of places "import" looks
# for modules, so this way it can find the data module.

import sys
sys.path.insert(0, "../") # replace with path/to/project/root
from data import get_data, get_site_ids

Site uids are the unique identifiers for water sites and their basins. We access the rest of the data by specifying one.

Let's see all of them:

In [ ]:
ids = get_site_ids()
print(f"Num sites: {len(ids)}")
print(ids[:5])

All sites follow the pattern `WQS*` (Iowa state sites) or `USGS-*` (USGS NWIS sites). Let's choose a USGS site (I picked one that I like) and look at its data.

In [ ]:
uid = "USGS-06604440"
data = get_data(site_uid=uid) # contains basin, crops, rain, surplus, water fields

Quick list of the contents:
| Value | Description |
|-----------|------------|
| `data.basin` | the basin of the site. Probably don't need to access this directly, used mostly to compute the other pieces of data. |
| `data.crop` | crop info for crops in the basin |
| `data.rain` | historical rain data for the basin from the site's start to end date. |
| `data.surplus` | nitrogen data inside the basin spanning 2000 till 2017. | 
| `data.water` | the water timeseries data for the site |

In [ ]:
data.water.info()

In [ ]:
# plot the water timeseries
import plotly.express as px

# the index of the water dataframe is the DateTime of the row
# its used as an index because it speeds up various DateTime operations

fig = px.line(
    data.water, 
    x=data.water.index, 
    y="nitrate_con")
fig.show()

Plotting the whole timeseries might make your notebook crash for long-lived sites because `data.water` is unaggregated; you're seeing the entire timeseries. Better to aggregate and then plot. Two options:

In [ ]:
# Option 1: aggregate manually
uid = "USGS-05482500" # 18 year lifespan, lots of data
water = get_data(site_uid=uid).water
nitrate = water["nitrate_con"].resample("1W").agg("mean")
    # aggregation easy bc the index is a datetime, just call resample and agg

fig = px.line(nitrate, x=nitrate.index, y="nitrate_con")
fig.show()


In [ ]:
# Option 2: use the aggregation function from the water module
from data.water import aggregate_by_interval

uid = "USGS-05482500"
nitrate_con = aggregate_by_interval(site_uid=uid, value_col="nitrate_con", interval="1W", agg_func="mean")

fig = px.line(nitrate, x=nitrate.index, y="nitrate_con")
fig.show()

## Example: aggregate rain data

You can aggregate rain data similarly (but I was dumb and forgot to make date the index of the rain dataframes by default) or use `data.rain.aggregate_by_interval`. Both are below:

In [ ]:
uid = "USGS-05482500"
rain = get_data(uid).rain
rain.info()

In [ ]:
# Option 1: Manually, note the annoying "set_index" -> "resample" -> "reset_index" yoga
agg_rain = rain.set_index("date").groupby(["lon", "lat"])["precip_in_1d"].resample("1W").agg("sum").reset_index().rename(columns={"precip_in_1d" : "precip_1w"})

print(agg_rain.info())

# Option 2: rain submodule
from data.rain import aggregate_by_interval as agg_rain_func

agg_rain2 = agg_rain_func(df=rain, interval="1W", agg_func="sum")

print(agg_rain2.info())

Here's a method for aggregating both water and rain at once.

In [ ]:
uid = "USGS-05482500"
data = get_data(uid)
agg_data = data.aggregate_by_interval(interval="1w")

print(agg_data.water.info())
print(agg_data.rain[["date", "precip_1w"]].info()) 
# notes the column name change!
# precip_in_1d -> precip_1w

Notice the rain is a way larger dataframe. This is because we're aggregating only across time, we still have a row for every `(lat,lon)` pair in the basin. This is good for training models -- more features -- but bad for plotting. Here's a function that produces a plotly figure with combined rain and water data:

In [ ]:
import plotly.graph_objects as go

# expects water and rain aggregated as above
def plot_rain_water(water, rain):
    fig = go.Figure()
    fig.add_trace(
        go.Bar(
            x=rain["date"],
            y=rain["precip_1w"],
            name="Precip (in)",
            yaxis="y2",
            marker_color="#3a94fa",
        )
    )
    fig.update_layout(
        yaxis2=dict(
            title="Precipitation (in)",
            overlaying="y",
            side="right",
            showgrid=False,
        ),
    )
    fig.add_trace(
        go.Scatter(
            x=water.index,
            y=water.values,
            name="N (mg/L)",
            yaxis="y1",
            marker_color="#0f8e5e",
        )
    )
    fig.update_layout(
        yaxis={"title": "Nitrate (mg/L)"},
        xaxis={"title": None},
        legend={"orientation": "h", "y": -0.15},
        margin={"t": 20, "b": 40, "l": 50, "r": 50},
    )
    return fig

Try this graph in two different scenarios, once on `agg_data` as present now, and once after averaging out the (lat, lon) values of the rain data:

In [ ]:
fig = plot_rain_water(agg_data.water, agg_data.rain)
fig.show()

In [ ]:
# group by date and average
water_df = agg_data.water
print(agg_data.rain[["lon","lat"]].nunique())
rain_df = agg_data.rain.groupby("date").agg("mean").reset_index()
print(rain_df[["lon","lat"]].nunique())

fig = plot_rain_water(water=water_df, rain=rain_df)
fig.show()

There is also `surplus` data, that's maybe easier to explore in the widget and by inspecting the `data.surplus` DataFrame.

In [ ]:
uid = "WQS0003"

data = get_data(uid)
print(data.crops.columns)
print(data.surplus.columns)
print(data.grid.columns)

In [ ]:
print(data.grid.dtypes)

In [ ]:
years = set(range(2000, 2026))
bad_uids = []
for uid in get_site_ids():
    crops = get_data(uid).crops
    if years != set(crops.year.unique()):
        bad_uids.append(uid)
        print(f" {uid} bad: {crops.year.unique()}")

crops.year.unique()